# STM32F429 Power Management (PWR) Tutorial for Beginners

This tutorial provides a comprehensive guide to understanding and using the power management features of STM32F429 microcontrollers. We'll cover everything from basic sleep modes to advanced power optimization techniques.

## Table of Contents
1. [Introduction to Power Management](#introduction)
2. [STM32F429 Power Overview](#power-overview)
3. [Hardware Setup](#hardware-setup)
4. [Software Dependencies](#software-setup)
5. [Basic Initialization](#basic-init)
6. [Sleep Modes](#sleep-modes)
7. [Stop Modes](#stop-modes)
8. [Standby Mode](#standby-mode)
9. [Voltage Regulation](#voltage-reg)
10. [Backup Domain](#backup-domain)
11. [PVD (Programmable Voltage Detector)](#pvd)
12. [Advanced Features](#advanced)
13. [Troubleshooting](#troubleshooting)

<a id='introduction'></a>
## 1. Introduction to Power Management

Power management is crucial for battery-powered and energy-efficient applications. The STM32F429 provides multiple low-power modes to minimize energy consumption while maintaining functionality.

### Key Concepts:
- **Active Mode**: Full operation, maximum power consumption
- **Sleep Mode**: CPU stopped, peripherals active
- **Stop Mode**: Most clocks stopped, SRAM retained
- **Standby Mode**: Lowest power, SRAM lost, backup domain active
- **Current Consumption**: Measured in µA (microamps)

### Power Saving Techniques:
- Clock gating (stopping unused clocks)
- Peripheral shutdown
- Voltage scaling
- Low-power modes
- Backup domain usage

### Applications:
- Battery-powered devices
- IoT sensors
- Wearable devices
- Energy monitoring
- Remote systems

<a id='power-overview'></a>
## 2. STM32F429 Power Overview

The STM32F429 features a sophisticated power management system with multiple operating modes and voltage regulators.

### Power Supply Domains:
- **VDD**: Main power supply (1.8V to 3.6V)
- **VDDA**: Analog power supply (1.8V to 3.6V)
- **VDD_BOR**: Backup domain power (1.65V to 3.6V)
- **VBAT**: Battery backup (1.65V to 3.6V)

### Voltage Regulators:
- **Main Regulator**: Supplies digital circuits
- **Low-Power Regulator**: Reduced power in Stop mode
- **Backup Regulator**: Powers backup domain

### Low-Power Modes:
- **Sleep Mode**: CPU stopped, all peripherals active
- **Stop Mode**: All clocks stopped, SRAM retained
- **Standby Mode**: Lowest power, only backup domain active

### Wakeup Sources:
- External interrupts (WKUP pin)
- RTC alarms
- IWDG reset
- NRST pin

### Current Consumption (Typical):
- **Run Mode**: 100-300mA (depending on clock speed)
- **Sleep Mode**: 10-50mA
- **Stop Mode**: 10-100µA
- **Standby Mode**: 1-10µA

<a id='hardware-setup'></a>
## 3. Hardware Setup

### Required Components:
- STM32F429 Discovery board
- Power supply (1.8V-3.6V)
- Optional: Battery for VBAT
- Optional: External wakeup button

### Pin Connections:

| Function | STM32F429 Pin | Description |
|----------|---------------|-------------|
| VDD | Multiple | Main power supply |
| VDDA | Multiple | Analog power supply |
| VDD_BOR | VDD | Backup domain power |
| VBAT | VBAT pin | Battery backup |
| WKUP | PA0 | Wakeup pin |
| NRST | NRST | Reset pin |

### Power Supply Requirements:
- **VDD**: 1.8V to 3.6V
- **VDDA**: Same as VDD ±0.3V
- **VBAT**: 1.65V to 3.6V (when VDD < VBAT)
- **Current**: Depends on operating mode

### Backup Battery Connection:
```c
// VBAT pin should be connected to a 3V coin cell battery
// through a diode for reverse polarity protection
//
// STM32 VBAT ----|>|---- Battery (+)
//                      ---- Battery (-)
```

### External Wakeup Circuit:
```c
// WKUP pin (PA0) can be connected to a push button
// Pull-down resistor recommended
//
// STM32 PA0 ---- Push Button ---- GND
//            |
//            --- 10kΩ pull-down to GND
```

### Important Notes:
- Ensure proper decoupling capacitors (10µF + 0.1µF)
- VBAT should be connected for backup domain functionality
- WKUP pin has internal pull-down resistor
- Power sequencing: VDDA should not exceed VDD by more than 0.3V

<a id='software-setup'></a>
## 4. Software Dependencies

### Required Files:
- `pwr.h` - Header file with function prototypes and types
- `pwr.c` - Main power management driver implementation
- STM32F4xx HAL Library (`stm32f4xx_hal.h`, `stm32f4xx_hal_pwr.h`)
- Standard C libraries: `stdint.h`, `stdbool.h`, `string.h`

### Include Path Setup:
```c
#include "pwr.h"
#include "stm32f4xx.h"  // HAL library
```

### Build Configuration:
- Add source files to project: `pwr.c`
- Include header paths: path to `pwr.h`
- Enable PWR peripheral in STM32CubeMX
- Configure backup domain if using RTC
- Enable wakeup pin if needed

### PWR Handle Declaration:
```c
// PWR module doesn't require a handle structure
// All functions are static and use HAL PWR functions internally
```

### Error Handling:
- All functions return `PWR_StatusTypeDef`
- Check return values: `PWR_OK`, `PWR_ERROR`, `PWR_INVALID_PARAM`, `PWR_TIMEOUT`
- Use logging for debugging power state transitions

<a id='basic-init'></a>
## 5. Basic Initialization

### PWR Configuration Structure:
```c
PWR_ConfigTypeDef pwr_config = {
    .enablePVD = true,                    // Enable voltage monitoring
    .pvdLevel = PWR_PVD_LEVEL_2V8,        // 2.8V threshold
    .enableBackupAccess = true,           // Enable backup domain
    .enableWakeupPin = true               // Enable WKUP pin
};
```

### Basic PWR Initialization:
```c
#include "pwr.h"

int main(void) {
    // Configure power management
    PWR_ConfigTypeDef config = {
        .enablePVD = true,
        .pvdLevel = PWR_PVD_LEVEL_2V9,
        .enableBackupAccess = true,
        .enableWakeupPin = false
    };
    
    // Initialize power management
    PWR_StatusTypeDef status = PWR_Init(&config);
    if (status != PWR_OK) {
        printf("PWR initialization failed\n");
        return -1;
    }
    
    printf("Power management initialized successfully\n");
    
    // Main application loop
    while (1) {
        // Application code here
    }
}
```

### Default Initialization:
```c
// Initialize with default settings
PWR_StatusTypeDef status = PWR_InitDefault();
if (status != PWR_OK) {
    printf("PWR default initialization failed\n");
    return -1;
}
```

### Understanding PWR States:
- **PWR_OK**: Operation completed successfully
- **PWR_ERROR**: General error occurred
- **PWR_INVALID_PARAM**: Invalid parameter provided
- **PWR_TIMEOUT**: Operation timed out
- **PWR_NOT_READY**: Power system not ready

### Initialization Checklist:
- PWR clock enabled
- Backup access configured
- Wakeup pin configured
- PVD configured if enabled
- Voltage regulator ready

<a id='sleep-modes'></a>
## 6. Sleep Modes

### Sleep Mode Overview:
Sleep mode stops the CPU while keeping all peripherals active. This provides moderate power savings while maintaining system responsiveness.

### Entering Sleep Mode:
```c
#include "pwr.h"

void enter_sleep_mode(void) {
    printf("Entering Sleep mode...\n");
    
    // Configure sleep mode
    PWR_SleepModeTypeDef sleep_mode = PWR_SLEEP_MODE_WFI;
    
    // Enter sleep mode
    PWR_StatusTypeDef status = PWR_EnterSleepMode(sleep_mode);
    if (status != PWR_OK) {
        printf("Failed to enter sleep mode\n");
        return;
    }
    
    // CPU will stop here and resume on interrupt
    printf("Woke up from Sleep mode\n");
}
```

### Sleep Mode with Timeout:
```c
void sleep_with_timeout(uint32_t timeout_ms) {
    // Set up a timer for wakeup
    // Configure TIM2 for timeout
    
    // Enter sleep mode
    PWR_EnterSleepMode(PWR_SLEEP_MODE_WFI);
    
    // Code resumes here after timeout or interrupt
}
```

### Sleep Mode Configuration:
```c
// Configure system for sleep
void configure_for_sleep(void) {
    // Disable unused peripherals to save power
    // Keep only essential peripherals active
    
    // Configure GPIO to low-power state
    // Disable unused clocks
    
    // Ready to enter sleep
}
```

### Wakeup from Sleep Mode:
Sleep mode can be exited by:
- **Any interrupt**: WFI mode
- **Any event**: WFE mode
- **System reset**: NRST pin

### Sleep Mode Current Consumption:
- **Normal Sleep**: ~10-50mA (depends on active peripherals)
- **With Peripherals Disabled**: ~5-20mA
- **Deep Sleep**: ~1-5mA (if supported)

### Sleep Mode Usage Example:
```c
int main(void) {
    // Initialize system
    System_Init();
    PWR_InitDefault();
    
    while (1) {
        // Do some work
        process_data();
        
        // Enter sleep to save power between tasks
        PWR_EnterSleepMode(PWR_SLEEP_MODE_WFI);
        
        // Resume after interrupt
    }
}
```

### Important Notes:
- All peripheral clocks remain active
- SRAM content is preserved
- GPIO states are maintained
- Wakeup is immediate (no startup time)
- Suitable for short idle periods

<a id='stop-modes'></a>
## 7. Stop Modes

### Stop Mode Overview:
Stop mode provides significant power savings by stopping all clocks while retaining SRAM content. Two regulator modes are available for different power/performance trade-offs.

### Stop Mode Entry:
```c
#include "pwr.h"

void enter_stop_mode(void) {
    printf("Entering Stop mode...\n");
    
    // Configure stop mode
    PWR_StopEntryTypeDef entry = PWR_STOP_ENTRY_WFI;
    PWR_RegulatorTypeDef regulator = PWR_REGULATOR_LOW_POWER;
    
    // Prepare for stop mode
    // Disable unused peripherals
    // Save critical data if needed
    
    // Enter stop mode
    PWR_StatusTypeDef status = PWR_EnterStopMode(entry, regulator);
    if (status != PWR_OK) {
        printf("Failed to enter stop mode\n");
        return;
    }
    
    // CPU will stop here and resume after wakeup
    // System clock needs to be reconfigured
    
    // Reconfigure system clock after wakeup
    SystemClock_Config();
    
    printf("Woke up from Stop mode\n");
}
```

### Stop Mode with RTC Wakeup:
```c
void stop_with_rtc_wakeup(uint32_t seconds) {
    // Configure RTC for wakeup
    RTC_ConfigureWakeupTimer(seconds);
    
    // Enable RTC wakeup
    PWR_EnableWakeupSource(PWR_SRC_RTC_WAKEUP);
    
    // Enter stop mode
    PWR_EnterStopMode(PWR_STOP_ENTRY_WFI, PWR_REGULATOR_LOW_POWER);
    
    // Code resumes here after RTC wakeup
    // Reconfigure clocks
    SystemClock_Config();
}
```

### Regulator Mode Selection:
```c
// Low power regulator (better for power saving)
PWR_EnterStopMode(PWR_STOP_ENTRY_WFI, PWR_REGULATOR_LOW_POWER);

// Main regulator (faster wakeup)
PWR_EnterStopMode(PWR_STOP_ENTRY_WFI, PWR_REGULATOR_ON);
```

### Stop Mode Wakeup Sources:
- **External interrupts**: GPIO interrupts
- **RTC events**: Alarm, wakeup timer, timestamp
- **IWDG reset**: Independent watchdog
- **WKUP pin**: Dedicated wakeup pin (PA0)

### Stop Mode Current Consumption:
- **Main Regulator**: ~10-50µA
- **Low-Power Regulator**: ~1-10µA
- **With RTC**: Additional ~1-2µA

### Stop Mode Configuration:
```c
void prepare_stop_mode(void) {
    // Disable HSE oscillator for lower power
    RCC_OscInitTypeDef osc_init = {0};
    osc_init.OscillatorType = RCC_OSCILLATORTYPE_HSE;
    osc_init.HSEState = RCC_HSE_OFF;
    HAL_RCC_OscConfig(&osc_init);
    
    // Configure GPIOs for low power
    GPIO_ConfigLowPower();
    
    // Disable unused peripherals
    Peripheral_DeInit();
}
```

### Wakeup Handling:
```c
void HAL_PWR_PVDCallback(void) {
    // Handle PVD interrupt in stop mode
    // This can wake the system
}

void HAL_RTC_AlarmAEventCallback(RTC_HandleTypeDef *hrtc) {
    // Handle RTC alarm wakeup
    // System will exit stop mode
}
```

### Stop Mode Best Practices:
- Use low-power regulator for maximum savings
- Configure all wakeup sources before entering
- Reconfigure clocks immediately after wakeup
- Minimize SRAM usage to reduce retention current
- Use RTC for timed wakeups

<a id='standby-mode'></a>
## 8. Standby Mode

### Standby Mode Overview:
Standby mode provides the lowest power consumption by shutting down almost everything. Only the backup domain remains active, and SRAM content is lost.

### Entering Standby Mode:
```c
#include "pwr.h"

void enter_standby_mode(void) {
    printf("Entering Standby mode...\n");
    
    // Save critical data to backup registers
    PWR_WriteBackupRegister(0, critical_data);
    
    // Configure wakeup sources
    PWR_EnableWakeupSource(PWR_SRC_WAKEUP_PIN);
    PWR_EnableWakeupSource(PWR_SRC_RTC_ALARM);
    
    // Clear wakeup flags
    PWR_ClearWakeupFlag();
    
    // Enter standby mode
    PWR_StatusTypeDef status = PWR_EnterStandbyMode();
    if (status != PWR_OK) {
        printf("Failed to enter standby mode\n");
        return;
    }
    
    // System will reset after wakeup from standby
    // Code execution starts from beginning
}
```

### Standby Mode with RTC Alarm:
```c
void standby_with_rtc_alarm(uint32_t alarm_time) {
    // Configure RTC alarm
    RTC_SetAlarm(alarm_time);
    
    // Enable RTC alarm wakeup
    PWR_EnableWakeupSource(PWR_SRC_RTC_ALARM);
    
    // Save state information
    PWR_WriteBackupRegister(0, SYSTEM_STATE_STANDBY);
    PWR_WriteBackupRegister(1, alarm_time);
    
    // Enter standby
    PWR_EnterStandbyMode();
    
    // System resets here after wakeup
}
```

### Wakeup Source Configuration:
```c
// Enable WKUP pin (PA0)
PWR_EnableWakeupSource(PWR_SRC_WAKEUP_PIN);

// Enable RTC alarm
PWR_EnableWakeupSource(PWR_SRC_RTC_ALARM);

// Enable RTC wakeup timer
PWR_EnableWakeupSource(PWR_SRC_RTC_WAKEUP);

// Enable RTC timestamp
PWR_EnableWakeupSource(PWR_SRC_RTC_TIMESTAMP);
```

### Backup Registers Usage:
```c
// Save system state before standby
#define STATE_REGISTER 0
#define DATA_REGISTER 1

void save_system_state(void) {
    PWR_WriteBackupRegister(STATE_REGISTER, current_state);
    PWR_WriteBackupRegister(DATA_REGISTER, important_data);
}

// Restore system state after wakeup
void restore_system_state(void) {
    uint32_t state = PWR_ReadBackupRegister(STATE_REGISTER);
    uint32_t data = PWR_ReadBackupRegister(DATA_REGISTER);
    
    if (state == SYSTEM_STATE_STANDBY) {
        // Resumed from standby, restore context
        current_state = data;
        printf("Resumed from standby mode\n");
    }
}
```

### Standby Mode Current Consumption:
- **Basic Standby**: ~1-5µA
- **With RTC**: ~2-10µA
- **With Backup RAM**: Additional ~1µA per 4KB

### Wakeup Detection:
```c
int main(void) {
    // Check if woke up from standby
    if (PWR_GetWakeupSource() != PWR_SRC_NONE) {
        printf("Woke up from standby mode\n");
        
        // Determine wakeup source
        PWR_WakeupSourceTypeDef source = PWR_GetWakeupSource();
        switch (source) {
            case PWR_SRC_WAKEUP_PIN:
                printf("Wakeup from WKUP pin\n");
                break;
            case PWR_SRC_RTC_ALARM:
                printf("Wakeup from RTC alarm\n");
                break;
            // Handle other sources
        }
        
        // Clear wakeup flag
        PWR_ClearWakeupFlag();
        
        // Restore system state
        restore_system_state();
    }
    
    // Normal initialization
    System_Init();
    PWR_InitDefault();
    
    // Main application
    while (1) {
        // Application code
    }
}
```

### Standby Mode Considerations:
- **SRAM Loss**: All variables are lost
- **Register Reset**: Most peripherals reset
- **System Reset**: Appears as power-on reset
- **Backup Domain**: RTC and backup registers preserved
- **Long Wakeup**: Slower startup time

### Standby Mode Best Practices:
- Save critical data to backup registers
- Use RTC for timed wakeups
- Implement proper wakeup detection
- Minimize backup register usage
- Consider capacitor on VBAT for clean resets

<a id='voltage-reg'></a>
## 9. Voltage Regulation

### Voltage Regulator Overview:
The STM32F429 has multiple voltage regulators for different power modes and performance requirements.

### Main Regulator Control:
```c
#include "pwr.h"

// Enable main regulator in low-power mode
PWR_StatusTypeDef status = PWR_EnableMainRegulatorLowVoltage();
if (status != PWR_OK) {
    printf("Failed to configure main regulator\n");
}

// Disable main regulator low-voltage mode
PWR_DisableMainRegulatorLowVoltage();
```

### Regulator Modes:
```c
// Configure regulator for run mode
PWR_SetRegulatorMode(PWR_REGULATOR_MODE_RUN);

// Configure regulator for low-power run mode
PWR_SetRegulatorMode(PWR_REGULATOR_MODE_LOWPOWER);

// Configure regulator for stop mode
PWR_SetRegulatorMode(PWR_REGULATOR_MODE_STOP);
```

### Voltage Scaling:
```c
// Scale voltage for different performance levels
typedef enum {
    PWR_SCALE1 = 0,  // Highest performance, highest power
    PWR_SCALE2,      // Medium performance
    PWR_SCALE3       // Low power mode
} PWR_VoltageScaleTypeDef;

// Set voltage scale
PWR_StatusTypeDef status = PWR_SetVoltageScale(PWR_SCALE2);
if (status != PWR_OK) {
    printf("Failed to set voltage scale\n");
}
```

### Regulator Monitoring:
```c
// Check regulator status
PWR_RegulatorStateTypeDef reg_state = PWR_GetRegulatorState();
switch (reg_state) {
    case PWR_REGULATOR_ON:
        printf("Main regulator ON\n");
        break;
    case PWR_REGULATOR_LOWPOWER:
        printf("Low-power regulator active\n");
        break;
    case PWR_REGULATOR_OFF:
        printf("Regulator OFF\n");
        break;
}
```

### Voltage Scaling Configuration:
```c
void configure_voltage_scaling(void) {
    // For high-performance tasks
    PWR_SetVoltageScale(PWR_SCALE1);
    SystemClock_Config();  // Reconfigure clocks for new voltage
    
    // Do high-performance work
    perform_computation();
    
    // Switch to low-power for idle periods
    PWR_SetVoltageScale(PWR_SCALE3);
    SystemClock_Config();  // Reconfigure for lower frequency
    
    // Enter low-power mode
    PWR_EnterSleepMode(PWR_SLEEP_MODE_WFI);
}
```

### Regulator Voltage Levels:
- **Scale 1**: Full voltage (1.8V-3.6V), max frequency
- **Scale 2**: Reduced voltage, medium frequency
- **Scale 3**: Minimum voltage, low frequency

### Flash Latency Considerations:
```c
// Adjust flash latency when changing voltage scale
void adjust_flash_latency(PWR_VoltageScaleTypeDef scale) {
    FLASH_LatencyTypeDef latency;
    
    switch (scale) {
        case PWR_SCALE1:
            latency = FLASH_LATENCY_5;  // Highest frequency
            break;
        case PWR_SCALE2:
            latency = FLASH_LATENCY_3;
            break;
        case PWR_SCALE3:
            latency = FLASH_LATENCY_1;  // Lowest frequency
            break;
    }
    
    HAL_FLASH_SetLatency(latency);
}
```

### Dynamic Voltage Scaling:
```c
void dynamic_voltage_scaling(void) {
    static PWR_VoltageScaleTypeDef current_scale = PWR_SCALE2;
    
    // Monitor system load
    uint32_t cpu_load = get_cpu_load_percentage();
    
    if (cpu_load > 80 && current_scale != PWR_SCALE1) {
        // High load - increase voltage/frequency
        PWR_SetVoltageScale(PWR_SCALE1);
        adjust_flash_latency(PWR_SCALE1);
        current_scale = PWR_SCALE1;
    } else if (cpu_load < 20 && current_scale != PWR_SCALE3) {
        // Low load - decrease voltage/frequency
        PWR_SetVoltageScale(PWR_SCALE3);
        adjust_flash_latency(PWR_SCALE3);
        current_scale = PWR_SCALE3;
    }
}
```

### Regulator Best Practices:
- Match voltage scale to performance requirements
- Adjust flash latency when changing scales
- Monitor temperature when using high performance
- Use dynamic scaling for optimal efficiency
- Test thoroughly when changing voltage levels

<a id='backup-domain'></a>
## 10. Backup Domain

### Backup Domain Overview:
The backup domain remains powered even when the main power is off, allowing data retention and RTC operation.

### Backup Register Usage:
```c
#include "pwr.h"

// Write data to backup register
PWR_StatusTypeDef status = PWR_WriteBackupRegister(0, 0x12345678);
if (status != PWR_OK) {
    printf("Failed to write backup register\n");
}

// Read data from backup register
uint32_t data = PWR_ReadBackupRegister(0);
printf("Backup register 0: 0x%08X\n", data);
```

### Backup Domain Access:
```c
// Enable backup domain access
PWR_EnableBackupAccess();

// Disable backup domain access (for security)
PWR_DisableBackupAccess();
```

### Backup Register Applications:
```c
// Store system configuration
#define REG_SYSTEM_STATE 0
#define REG_BOOT_COUNT 1
#define REG_LAST_ERROR 2

void save_system_config(void) {
    PWR_WriteBackupRegister(REG_SYSTEM_STATE, current_state);
    PWR_WriteBackupRegister(REG_BOOT_COUNT, boot_count++);
    PWR_WriteBackupRegister(REG_LAST_ERROR, last_error_code);
}

void load_system_config(void) {
    current_state = PWR_ReadBackupRegister(REG_SYSTEM_STATE);
    boot_count = PWR_ReadBackupRegister(REG_BOOT_COUNT);
    last_error_code = PWR_ReadBackupRegister(REG_LAST_ERROR);
}
```

### Backup Domain Reset:
```c
// Reset backup domain (clears all backup registers)
PWR_ResetBackupDomain();

// Check if backup domain was reset
bool is_reset = PWR_IsBackupDomainReset();
if (is_reset) {
    printf("Backup domain was reset\n");
    // Initialize backup registers
    initialize_backup_data();
}
```

### Backup SRAM Usage:
```c
// Enable backup SRAM clock
__HAL_RCC_BKPSRAM_CLK_ENABLE();

// Access backup SRAM (4KB available)
#define BACKUP_SRAM_START 0x40024000
uint32_t* backup_sram = (uint32_t*)BACKUP_SRAM_START;

// Write to backup SRAM
backup_sram[0] = sensor_data;
backup_sram[1] = timestamp;

// Read from backup SRAM
uint32_t saved_data = backup_sram[0];
uint32_t saved_time = backup_sram[1];
```

### Backup Domain Protection:
```c
// Enable tamper detection
PWR_EnableTamperDetection();

// Disable tamper detection
PWR_DisableTamperDetection();

// Check tamper status
bool tamper_detected = PWR_IsTamperDetected();
if (tamper_detected) {
    printf("Tamper detected - backup domain compromised\n");
    // Take security measures
    security_alert();
}
```

### RTC Integration:
```c
// Configure RTC in backup domain
RTC_ConfigTypeDef rtc_config = {
    .hour_format = RTC_HOURFORMAT_24,
    .asynch_prediv = 127,
    .synch_prediv = 255
};

PWR_EnableBackupAccess();
RTC_Init(&hrtc, &rtc_config);

// RTC continues running even in standby mode
```

### Backup Domain Power:
```c
// Check backup domain power status
PWR_BackupPowerTypeDef backup_power = PWR_GetBackupPowerState();
switch (backup_power) {
    case PWR_BACKUP_POWER_ON:
        printf("Backup domain powered from VDD\n");
        break;
    case PWR_BACKUP_POWER_BATTERY:
        printf("Backup domain powered from VBAT\n");
        break;
    case PWR_BACKUP_POWER_OFF:
        printf("Backup domain not powered\n");
        break;
}
```

### Backup Domain Best Practices:
- Use backup registers for critical system state
- Enable backup access only when needed
- Implement tamper detection for security
- Monitor backup power source
- Initialize backup data after reset
- Use backup SRAM for larger data storage

<a id='pvd'></a>
## 11. PVD (Programmable Voltage Detector)

### PVD Overview:
The Programmable Voltage Detector monitors the VDD voltage and can generate interrupts or events when voltage drops below configurable thresholds.

### PVD Configuration:
```c
#include "pwr.h"

// Configure PVD with 2.8V threshold
PWR_ConfigTypeDef config = {
    .enablePVD = true,
    .pvdLevel = PWR_PVD_LEVEL_2V8,
    .enableBackupAccess = true,
    .enableWakeupPin = false
};

PWR_StatusTypeDef status = PWR_Init(&config);
if (status != PWR_OK) {
    printf("PVD configuration failed\n");
}
```

### PVD Threshold Levels:
```c
// Available PVD thresholds
PWR_EnablePVD(PWR_PVD_LEVEL_2V0);  // 2.0V
PWR_EnablePVD(PWR_PVD_LEVEL_2V1);  // 2.1V
PWR_EnablePVD(PWR_PVD_LEVEL_2V3);  // 2.3V
PWR_EnablePVD(PWR_PVD_LEVEL_2V5);  // 2.5V
PWR_EnablePVD(PWR_PVD_LEVEL_2V6);  // 2.6V
PWR_EnablePVD(PWR_PVD_LEVEL_2V7);  // 2.7V
PWR_EnablePVD(PWR_PVD_LEVEL_2V8);  // 2.8V
PWR_EnablePVD(PWR_PVD_LEVEL_2V9);  // 2.9V
```

### PVD Interrupt Handling:
```c
// PVD interrupt callback
void HAL_PWR_PVDCallback(void) {
    printf("PVD interrupt: Voltage dropped below threshold\n");
    
    // Take emergency actions
    emergency_shutdown();
    
    // Or prepare for power failure
    save_critical_data();
}

// Enable PVD interrupt
void enable_pvd_interrupt(void) {
    // Enable PVD
    PWR_EnablePVD(PWR_PVD_LEVEL_2V5);
    
    // Enable PVD interrupt in NVIC
    HAL_NVIC_EnableIRQ(PVD_IRQn);
    
    // Set interrupt priority
    HAL_NVIC_SetPriority(PVD_IRQn, 0, 0);
}
```

### PVD Status Monitoring:
```c
// Check PVD status
bool pvd_triggered = PWR_IsPVDTriggered();
if (pvd_triggered) {
    printf("PVD triggered - low voltage detected\n");
    
    // Handle low voltage condition
    handle_low_voltage();
}

// Get current PVD threshold
PWR_PVDLevelTypeDef current_level = PWR_GetPVDLevel();
printf("Current PVD threshold: ");
switch (current_level) {
    case PWR_PVD_LEVEL_2V0: printf("2.0V\n"); break;
    case PWR_PVD_LEVEL_2V5: printf("2.5V\n"); break;
    // Add other cases
}
```

### PVD Applications:
```c
// Battery monitoring
void monitor_battery_voltage(void) {
    static bool low_voltage_warning = false;
    
    if (PWR_IsPVDTriggered() && !low_voltage_warning) {
        printf("WARNING: Battery voltage low!\n");
        low_voltage_warning = true;
        
        // Reduce system activity
        reduce_system_load();
        
        // Alert user
        battery_warning_led(true);
    } else if (!PWR_IsPVDTriggered() && low_voltage_warning) {
        printf("Battery voltage restored\n");
        low_voltage_warning = false;
        battery_warning_led(false);
    }
}

// Emergency shutdown
void emergency_shutdown(void) {
    printf("Emergency shutdown initiated\n");
    
    // Save critical data
    save_system_state();
    
    // Stop all activities
    stop_all_tasks();
    
    // Enter standby mode
    PWR_EnterStandbyMode();
}
```

### PVD with Stop Mode:
```c
// Use PVD to wake from stop mode on voltage recovery
void pvd_stop_mode_demo(void) {
    // Configure PVD
    PWR_EnablePVD(PWR_PVD_LEVEL_2V5);
    
    // Configure PVD to generate event (not interrupt)
    PWR_PVDModeTypeDef pvd_mode = PWR_PVD_MODE_EVENT;
    PWR_SetPVDMode(pvd_mode);
    
    printf("Entering stop mode with PVD monitoring\n");
    
    // Enter stop mode
    PWR_EnterStopMode(PWR_STOP_ENTRY_WFE, PWR_REGULATOR_LOW_POWER);
    
    // System wakes up when voltage recovers
    SystemClock_Config();
    printf("Woke up from stop mode - voltage recovered\n");
}
```

### PVD Best Practices:
- Choose appropriate threshold for your application
- Implement proper interrupt handling
- Use for battery monitoring and emergency shutdown
- Combine with low-power modes for power management
- Test PVD functionality with variable power supply
- Consider hysteresis to prevent oscillations

<a id='advanced'></a>
## 12. Advanced Features

### Power Monitoring and Statistics:
```c
#include "pwr.h"

// Get power consumption statistics
PWR_StatsTypeDef stats;
PWR_GetPowerStats(&stats);

printf("Power Statistics:\n");
printf("Sleep mode entries: %lu\n", stats.sleep_entries);
printf("Stop mode entries: %lu\n", stats.stop_entries);
printf("Standby mode entries: %lu\n", stats.standby_entries);
printf("Average current: %.2f mA\n", stats.avg_current_ma);
printf("Total energy used: %.2f mWh\n", stats.total_energy_mwh);
```

### Advanced Sleep Configuration:
```c
// Configure sleep mode with specific peripherals enabled
PWR_SleepConfigTypeDef sleep_config = {
    .keep_peripherals = PWR_PERIPH_TIM2 | PWR_PERIPH_USART1,
    .disable_clocks = PWR_CLOCK_HSI | PWR_CLOCK_HSE,
    .gpio_power_down = true,
    .flash_power_down = false
};

PWR_ConfigureSleepMode(&sleep_config);
PWR_EnterSleepMode(PWR_SLEEP_MODE_WFI);
```

### Multi-Mode Power Management:
```c
typedef enum {
    POWER_MODE_HIGH_PERF,
    POWER_MODE_BALANCED,
    POWER_MODE_LOW_POWER,
    POWER_MODE_ULTRA_LOW
} PowerModeTypeDef;

void set_power_mode(PowerModeTypeDef mode) {
    switch (mode) {
        case POWER_MODE_HIGH_PERF:
            PWR_SetVoltageScale(PWR_SCALE1);
            PWR_SetRegulatorMode(PWR_REGULATOR_MODE_RUN);
            SystemClock_Config_High();  // 180MHz
            break;
            
        case POWER_MODE_BALANCED:
            PWR_SetVoltageScale(PWR_SCALE2);
            PWR_SetRegulatorMode(PWR_REGULATOR_MODE_RUN);
            SystemClock_Config_Medium();  // 120MHz
            break;
            
        case POWER_MODE_LOW_POWER:
            PWR_SetVoltageScale(PWR_SCALE3);
            PWR_SetRegulatorMode(PWR_REGULATOR_MODE_LOWPOWER);
            SystemClock_Config_Low();  // 16MHz
            break;
            
        case POWER_MODE_ULTRA_LOW:
            // Prepare for standby
            PWR_EnableWakeupSource(PWR_SRC_RTC_WAKEUP);
            PWR_EnterStandbyMode();
            break;
    }
}
```

### Temperature Monitoring:
```c
// Monitor internal temperature sensor
float get_internal_temperature(void) {
    // Enable temperature sensor
    ADC_EnableTempSensor();
    
    // Read temperature
    uint16_t adc_value = ADC_ReadTempSensor();
    
    // Convert to Celsius
    float temperature = ADC_ConvertToTemperature(adc_value);
    
    return temperature;
}

// Temperature-based power management
void temperature_power_control(void) {
    float temp = get_internal_temperature();
    
    if (temp > 85.0f) {
        // Overheating - reduce performance
        set_power_mode(POWER_MODE_LOW_POWER);
        printf("Overheating detected: %.1f°C - reducing performance\n", temp);
    } else if (temp < 60.0f) {
        // Cool - can increase performance
        set_power_mode(POWER_MODE_BALANCED);
    }
}
```

### Smart Wakeup System:
```c
// Configure multiple wakeup sources with priorities
PWR_WakeupConfigTypeDef wakeup_config = {
    .sources = PWR_SRC_WAKEUP_PIN | PWR_SRC_RTC_ALARM,
    .pin_priority = PWR_PRIORITY_HIGH,
    .rtc_priority = PWR_PRIORITY_MEDIUM,
    .timeout_ms = 30000  // 30 second timeout
};

PWR_ConfigureWakeupSystem(&wakeup_config);

// Enter smart sleep
PWR_EnterSmartSleepMode();

// System will wake on highest priority source
PWR_WakeupSourceTypeDef wakeup_source = PWR_GetWakeupSource();
printf("Woke up from source: %d\n", wakeup_source);
```

### Power Failure Recovery:
```c
// Implement power failure detection and recovery
void power_failure_recovery(void) {
    // Check if power was restored after failure
    if (PWR_WasPowerFailure()) {
        printf("Power failure recovery initiated\n");
        
        // Check backup registers for recovery information
        uint32_t recovery_state = PWR_ReadBackupRegister(0);
        
        switch (recovery_state) {
            case RECOVERY_FROM_SLEEP:
                // Resume from sleep
                restore_sleep_context();
                break;
            case RECOVERY_FROM_STOP:
                // Resume from stop
                SystemClock_Config();
                restore_stop_context();
                break;
            case RECOVERY_FROM_STANDBY:
                // Full system restart
                system_cold_start();
                break;
        }
        
        // Clear power failure flag
        PWR_ClearPowerFailureFlag();
    }
}
```

### Energy Harvesting Support:
```c
// Support for energy harvesting applications
void energy_harvesting_mode(void) {
    // Configure for ultra-low power
    PWR_SetVoltageScale(PWR_SCALE3);
    PWR_SetRegulatorMode(PWR_REGULATOR_MODE_LOWPOWER);
    
    // Enable PVD for minimum voltage detection
    PWR_EnablePVD(PWR_PVD_LEVEL_1V8);
    
    // Configure wakeup on capacitor charge
    PWR_EnableWakeupSource(PWR_SRC_WAKEUP_PIN);
    
    // Enter burst mode: sleep, wake, process, sleep
    while (1) {
        PWR_EnterSleepMode(PWR_SLEEP_MODE_WFI);
        
        // Quick processing burst
        process_data_burst();
        
        // Back to sleep
        if (energy_insufficient()) {
            PWR_EnterStandbyMode();
        }
    }
}
```

### Debug and Monitoring:
```c
// Advanced power debugging
void power_debug_monitor(void) {
    PWR_DebugInfoTypeDef debug_info;
    PWR_GetDebugInfo(&debug_info);
    
    printf("Power Debug Information:\n");
    printf("Current mode: %s\n", PWR_GetModeString(debug_info.current_mode));
    printf("Voltage scale: %d\n", debug_info.voltage_scale);
    printf("Regulator state: %s\n", PWR_GetRegulatorString(debug_info.regulator_state));
    printf("Wakeup sources: 0x%04X\n", debug_info.wakeup_sources);
    printf("PVD threshold: %.1fV\n", PWR_GetPVDVoltage(debug_info.pvd_level));
    printf("Backup registers used: %d\n", debug_info.backup_regs_used);
}
```

### Performance Optimization:
- **Dynamic Voltage Scaling**: Adjust voltage based on load
- **Peripheral Clock Gating**: Disable unused peripheral clocks
- **Flash Optimization**: Adjust wait states for voltage/frequency
- **Memory Optimization**: Use appropriate memory regions
- **Interrupt Optimization**: Minimize interrupt latency

<a id='troubleshooting'></a>
## 13. Troubleshooting

### Common Power Management Issues:

#### 1. System Won't Enter Low-Power Mode
**Symptoms:** PWR_EnterSleepMode() returns immediately without entering sleep
**Possible Causes:**
- Interrupts not properly configured
- Pending interrupts preventing sleep
- Incorrect PWR configuration
- Debug mode preventing sleep
**Solutions:**
- Clear all pending interrupts before entering sleep
- Disable debug features that prevent sleep
- Check PWR configuration parameters
- Verify interrupt priorities

#### 2. High Current Consumption in Sleep Mode
**Symptoms:** Current consumption higher than expected in sleep mode
**Possible Causes:**
- Peripherals not disabled
- GPIO configurations not optimized
- Clocks not stopped
- Debug interface active
**Solutions:**
- Disable unused peripherals before sleep
- Configure GPIOs for low-power state
- Stop unused clocks
- Disable debug interfaces

#### 3. System Won't Wake from Stop Mode
**Symptoms:** System enters stop mode but doesn't wake up
**Possible Causes:**
- Wakeup sources not configured
- Incorrect wakeup pin configuration
- RTC not configured for wakeup
- Clock configuration lost
**Solutions:**
- Verify wakeup source configuration
- Check WKUP pin connection and configuration
- Ensure RTC is properly configured
- Reconfigure clocks after wakeup

#### 4. Backup Registers Not Retaining Data
**Symptoms:** Backup register values lost after reset
**Possible Causes:**
- VBAT not connected or insufficient voltage
- Backup domain reset
- Tamper detection triggered
- Backup access not enabled
**Solutions:**
- Check VBAT voltage (should be >1.65V)
- Verify backup domain power connection
- Check for tamper events
- Enable backup access before writing

#### 5. PVD Not Triggering
**Symptoms:** Voltage drops but PVD interrupt doesn't occur
**Possible Causes:**
- PVD not enabled
- Incorrect threshold configuration
- PVD interrupt not enabled in NVIC
- Voltage drop too fast
**Solutions:**
- Enable PVD with PWR_EnablePVD()
- Configure appropriate threshold
- Enable PVD_IRQn in NVIC
- Use slower voltage ramp for testing

#### 6. Standby Mode Issues
**Symptoms:** System doesn't enter standby or behaves unexpectedly
**Possible Causes:**
- Wakeup sources not cleared
- Backup domain not configured
- Critical data not saved
- System appears to reset unexpectedly
**Solutions:**
- Clear wakeup flags before entering standby
- Save critical data to backup registers
- Implement proper wakeup detection
- Handle standby as system reset

#### 7. Voltage Scaling Problems
**Symptoms:** System crashes when changing voltage scale
**Possible Causes:**
- Flash latency not adjusted
- Clock frequency too high for voltage
- PLL configuration invalid
- Interrupt context switching
**Solutions:**
- Adjust flash latency for new voltage scale
- Ensure clock frequency is valid for voltage
- Reconfigure PLL after voltage change
- Avoid voltage scaling in interrupt handlers

### Debug Tips:
```c
// Check PWR peripheral status
PWR_StatusTypeDef status = PWR_GetStatus();
printf("PWR Status: %s\n", PWR_GetStatusString(status));

// Monitor power mode transitions
PWR_ModeTypeDef current_mode = PWR_GetCurrentMode();
printf("Current power mode: %s\n", PWR_GetModeString(current_mode));

// Check wakeup sources
PWR_WakeupSourceTypeDef sources = PWR_GetWakeupSource();
printf("Wakeup sources: 0x%04X\n", sources);

// Verify backup register access
PWR_EnableBackupAccess();
uint32_t test_value = 0xDEADBEEF;
PWR_WriteBackupRegister(0, test_value);
uint32_t read_value = PWR_ReadBackupRegister(0);
if (read_value == test_value) {
    printf("Backup register test passed\n");
} else {
    printf("Backup register test failed\n");
}

// Check PVD status
if (PWR_IsPVDEnabled()) {
    printf("PVD enabled, threshold: %s\n", PWR_GetPVDLevelString(PWR_GetPVDLevel()));
} else {
    printf("PVD disabled\n");
}

// Monitor voltage levels (if ADC available)
float vdd_voltage = ADC_ReadVDDVoltage();
printf("VDD voltage: %.2fV\n", vdd_voltage);

// Check regulator status
PWR_RegulatorStateTypeDef reg_state = PWR_GetRegulatorState();
printf("Regulator state: %s\n", PWR_GetRegulatorString(reg_state));
```

### Hardware Debugging:
- **Multimeter**: Measure current consumption in different modes
- **Oscilloscope**: Check voltage levels and wakeup signals
- **Logic Analyzer**: Monitor WKUP pin and interrupt signals
- **Power Supply**: Variable supply for testing PVD thresholds
- **Battery Simulator**: Test backup domain functionality

### Common Mistakes:
- Forgetting to reconfigure clocks after stop mode wakeup
- Not saving critical data before standby mode
- Incorrect PVD threshold for application voltage range
- Leaving debug interfaces enabled in production
- Not handling all wakeup sources properly
- Assuming SRAM content is preserved in standby mode

### Performance Verification:
```c
// Measure current consumption
void measure_current_consumption(void) {
    // Configure for measurement
    PWR_SetVoltageScale(PWR_SCALE3);
    disable_all_peripherals();
    
    // Enter sleep mode
    PWR_EnterSleepMode(PWR_SLEEP_MODE_WFI);
    
    // External measurement equipment measures current
    // Should be < 10mA for sleep mode
}

// Test wakeup latency
void test_wakeup_latency(void) {
    uint32_t start_time = HAL_GetTick();
    
    // Enter stop mode
    PWR_EnterStopMode(PWR_STOP_ENTRY_WFI, PWR_REGULATOR_LOW_POWER);
    
    // Measure wakeup time
    uint32_t wakeup_time = HAL_GetTick();
    uint32_t latency = wakeup_time - start_time;
    
    printf("Stop mode wakeup latency: %lu ms\n", latency);
}
```

### Getting Help:
- Check STM32F429 Reference Manual (PWR chapter)
- Review HAL PWR driver documentation
- Verify power supply requirements
- Test with known working power configurations
- Check errata sheet for silicon revisions
- Consult STMicroelectronics forums for similar issues

For more detailed information, refer to the STM32F429 datasheet, reference manual, and power management application notes.